Genetic status cohort definition analysis in GP2 Neurobooster genotyping data (all ancestries)

Project: GP2 lysosomal PRS

Version: Python/3.10.17, R/4.4.2

Notebook Overview

1. Description Loading Python libraries Set paths Make working directory

2. Installing packages

3. process clinical data

4. GBA1 cohort definition

5. LRRK2 and other known PD pathogenic genes cohort definition

6. Cohort sorting

Loading Python libraries

In [ ]:
# Use pathlib for file path manipulation
import pathlib

# Install numpy
import numpy as np

# Install Pandas for tabular data
import pandas as pd

# Install plotnine: a ggplot2-compatible Python plotting package
from plotnine import *

# Always show all columns in a Pandas DataFrame
pd.set_option('display.max_columns', None)

Set paths

In [ ]:
WORK_DIR = "~/workspace/ws_files/"

In [ ]:
REL11_PATH = pathlib.Path(pathlib.Path.home(), 'workspace/gp2_tier2_eu_release11')
!ls -hal {REL11_PATH}

install packages

PLINK

In [ ]:
%%bash

mkdir -p ~/tools
cd ~/tools

if test -e /home/jupyter/tools/plink; then
echo "Plink1.9 is already installed in /home/jupyter/tools/"

else
echo -e "Downloading plink \n    -------"
wget -N http://s3.amazonaws.com/plink1-assets/plink_linux_x86_64_20190304.zip 
unzip -o plink_linux_x86_64_20190304.zip
echo -e "\n plink downloaded and unzipped in /home/jupyter/tools \n "

fi

In [ ]:
%%capture
%%bash

# Install plink 2.0
cd /home/jupyter/tools/
if test -e /home/jupyter/tools/plink2; then

echo "Plink2 is already installed in /home/jupyter/tools/"
else
echo "Plink2 is not installed"
cd /home/jupyter/tools/

wget http://s3.amazonaws.com/plink2-assets/plink2_linux_x86_64_latest.zip

unzip -o plink2_linux_x86_64_latest.zip

fi

In [ ]:
%%bash

# Install ANNOVAR:
# https://www.openbioinformatics.org/annovar/annovar_download_form.php

if test -e /home/jupyter/tools/annovar; then

echo "annovar is already installed in /home/jupyter/tools/"
else
echo "annovar is not installed"
cd /home/jupyter/tools/

wget http://www.openbioinformatics.org/annovar/download/0wgxR2rIVP/annovar.latest.tar.gz

tar xvfz annovar.latest.tar.gz

fi

Install ANNOVAR: Download sources of annotation

In [ ]:
%%bash

cd /home/jupyter/tools/annovar/

perl annotate_variation.pl -buildver hg38 -downdb -webfrom annovar refGene humandb/
perl annotate_variation.pl -buildver hg38 -downdb -webfrom annovar clinvar_20140902 humandb/
#perl annotate_variation.pl -buildver hg38 -downdb cytoBand humandb/
#perl annotate_variation.pl -buildver hg38 -downdb -webfrom annovar ensGene humandb/
#perl annotate_variation.pl -buildver hg38 -downdb -webfrom annovar exac03 humandb/ 
#perl annotate_variation.pl -buildver hg38 -downdb -webfrom annovar avsnp147 humandb/ 
#perl annotate_variation.pl -buildver hg38 -downdb -webfrom annovar dbnsfp30a humandb/
#perl annotate_variation.pl -buildver hg38 -downdb -webfrom annovar gnomad211_genome humandb/
#perl annotate_variation.pl -buildver hg38 -downdb -webfrom annovar ljb26_all humandb/

In [ ]:
! ls /home/jupyter/tools

process clinical data

In [ ]:
CLINICAL_DATA_PATH = pathlib.Path(REL11_PATH, 'clinical_data/master_key_release11_final_vwb.csv')

In [ ]:
# Let's load the master key
key = pd.read_csv(CLINICAL_DATA_PATH, low_memory=False)
print(key.shape)
key.head()

In [ ]:
# look at sample size of each cohort

(key['study'] == 'TUEPAC').sum()

In [ ]:
# filter out european population firstly
EUR = key[key['nba_label'] == 'EUR']
EUR

In [ ]:
# filter out european population firstly
FIN = key[key['nba_label'] == 'FIN']
FIN

In [ ]:
# Subsetting to keep only a few columns 
key = key[['GP2ID', 'baseline_GP2_phenotype_for_qc', 'biological_sex_for_qc', 'age_at_sample_collection', 'age_of_onset', 'nba_label']]
# Renaming the columns
key.rename(columns = {'GP2ID':'IID',
                                     'baseline_GP2_phenotype_for_qc':'phenotype',
                                     'biological_sex_for_qc':'SEX', 
                                     'age_at_sample_collection':'AGE', 
                                     'age_of_onset':'AAO'}, inplace = True)
key

In [ ]:
%%bash
# making working directory
#Loop over all the ancestries
for ancestry in {'EUR','AAC','AFR','AJ','AMR','CAS','EAS','MDE','SAS','CAH'} ;
do

#Make a folder for each ancestry
mkdir ~/workspace/ws_files/r11/cohort/"$ancestry"

done

In [ ]:
ancestries = {'EUR','AAC','AFR','AJ','AMR','SAS','EAS','CAS','MDE','CAH'}

for ancestry in ancestries:
    
    WORK_DIR = f'~/workspace/ws_files/r11/cohort/{ancestry}'
    print(f'WORKING ON: {ancestry}')
    
    ## Subset to keep ancestry of interest 
    ancestry_key = key[key['nba_label']==ancestry].copy()
    ancestry_key.reset_index(drop=True)
    
     # Load information about related individuals in the ancestry analyzed
    related_df = pd.read_csv(f'{REL11_PATH}/meta_data/related_samples/{ancestry}_release11_vwb.related')
    print(f'Related individuals: {related_df.shape}')
    
    # Make a list of just one set of related people
    related_list = list(related_df['IID1'])
    
    # Check value counts of related and remove only one related individual
    ancestry_key = ancestry_key[~ancestry_key["IID"].isin(related_list)]
    
    # Check size
    print(f'Unrelated individuals: {ancestry_key.shape}')
    
    # Convert phenotype to binary (1/2)
    ## Assign conditions so case=2 and controls=1, and -9 otherwise (matching PLINK convention)
    # PD = 2; control = 1
    pheno_mapping = {"PD": 2, "Control": 1}
    ancestry_key['PHENO'] = ancestry_key['phenotype'].map(pheno_mapping).astype('Int64')
    
    # Check value counts of pheno
    ancestry_key['PHENO'].value_counts(dropna=False)
    
    # Check value counts of SEX
    sex_og_values = ancestry_key['SEX'].value_counts(dropna=False)
    print(f'Sex value counts - original:\n {sex_og_values.to_string()}')
    
     # Convert sex to binary (1/2)
    ## Assign conditions so female=2 and men=1, and -9 otherwise (matching PLINK convention)
    # Female = 2; Male = 1
    sex_mapping = {"Female": 2, "Male": 1}
    ancestry_key['SEX'] = ancestry_key['SEX'].map(sex_mapping).astype('Int64')
    
    # Check value counts of SEX after recoding
    sex_recode_values = ancestry_key['SEX'].value_counts(dropna=False)
    print(f'Sex value counts - recoded:\n{sex_recode_values.to_string()}')
    
    #Make additional columns - FID, fatid and matid - these are needed for RVtests!!
    #RVtests needs the first 5 columns to be fid, iid, fatid, matid and sex otherwise it does not run correctly
    #Uppercase column name is ok
    #See https://zhanxw.github.io/rvtests/#phenotype-file
    ancestry_key['FID'] = 0
    ancestry_key['FATID'] = 0
    ancestry_key['MATID'] = 0
    
    ## Clean up and keep columns we need 
    final_df = ancestry_key[['FID','IID', 'FATID', 'MATID', 'SEX', 'AGE', 'PHENO']].copy()
    final_df['FID'] = final_df['IID']
    
    ##DO NOT replace missing values with -9 as this is misinterpreted by RVtests - needs to be nonnumeric
    #Leave missing values as NA
    
    #Check number of PD cases missing age
    pd_missAge = final_df[(final_df['PHENO']==2)&(final_df['AGE'].isna())]
    print(f'Number of PD cases missing age: {pd_missAge.shape[0]}')
    
    #Check number of controls missing age
    control_missAge = final_df[(final_df['PHENO']==1)&(final_df['AGE'].isna())]
    print(f'Number of controls missing age: {control_missAge.shape[0]}')
    
    ## Make file of sample IDs to keep 
    samples_toKeep = final_df[['FID', 'IID']].copy()
    samples_toKeep.to_csv(f'~/workspace/ws_files/r11/cohort/{ancestry}/{ancestry}.samplestoKeep', sep = '\t', index=False, header=None)
    
    ## Make your covariate file
    #Included na_rep to write out missing/NA values explicitly as string/text, not as blank otherwise they are misread in RVtests
    final_df.to_csv(f'~/workspace/ws_files/r11/cohort/{ancestry}/{ancestry}_covariate_file.txt', sep = '\t', na_rep='NA', index=False)

GBA1 cohort definition

Lara's file to define GBA1 cohort

In [ ]:
# read gba1_lrrk2 ccarriers file
carriers_ancestry = pd.read_csv("~/workspace/ws_files/r11/results/GBA1_LRRK2_check/carriers_ancestry.csv", sep=",")
carriers_ancestry

In [ ]:
# keep GBA1 risk carriers
GBA1_risk = carriers_ancestry[
    carriers_ancestry['carrier_group'].isin(['GBA1_only', 'GBA1_LRRK2'])
].copy()
GBA1_risk

In [ ]:
# remove missing ancestry or GP2ID
GBA1_risk = GBA1_risk.dropna(subset=['Ancestry', 'GP2ID'])
GBA1_risk

In [ ]:
# clean dataframe
GBA1_risk_clean = GBA1_risk.copy()

# remove old index column if present
GBA1_risk_clean = GBA1_risk_clean.drop(columns=['Unnamed: 0'], errors='ignore')

# remove missing GP2ID or Ancestry
GBA1_risk_clean = GBA1_risk_clean.dropna(subset=['GP2ID', 'Ancestry'])

# remove duplicated GP2ID within each ancestry
GBA1_risk_clean = GBA1_risk_clean.drop_duplicates(subset=['Ancestry', 'GP2ID'])

GBA1_risk_clean

In [ ]:
GBA1_risk_clean['Ancestry'].value_counts()

In [ ]:
import os
# base output directory
base_dir = os.path.expanduser("~/workspace/ws_files/r11/cohort")

# save one file for each ancestry
for ancestry, df_anc in GBA1_risk_clean.groupby('Ancestry'):

    # create ancestry folder
    out_dir = os.path.join(base_dir, ancestry)
    os.makedirs(out_dir, exist_ok=True)

    # make FID and IID columns from GP2ID
    samples_toKeep = pd.DataFrame({
        'FID': df_anc['GP2ID'],
        'IID': df_anc['GP2ID']
    })

    # output file
    out_file = os.path.join(out_dir, "GBA1risk.samplestoKeep.rm.txt")

    # save without header and without index
    samples_toKeep.to_csv(
        out_file,
        sep='\t',
        index=False,
        header=False
    )

    print(f"{ancestry}: {samples_toKeep.shape[0]} samples saved to {out_file}")

LRRK2 cohort definition

Lara's file to define LRRK2 cohort

In [ ]:
# keep LRRK2 risk carriers
LRRK2_risk = carriers_ancestry[
    carriers_ancestry['carrier_group'].isin(['LRRK2_only', 'GBA1_LRRK2'])
].copy()
LRRK2_risk

In [ ]:
# remove missing ancestry or GP2ID
LRRK2_risk = LRRK2_risk.dropna(subset=['Ancestry', 'GP2ID'])
LRRK2_risk

In [ ]:
# clean dataframe
LRRK2_risk_clean = LRRK2_risk.copy()

# remove old index column if present
LRRK2_risk_clean = LRRK2_risk_clean.drop(columns=['Unnamed: 0'], errors='ignore')

# remove missing GP2ID or Ancestry
LRRK2_risk_clean = LRRK2_risk_clean.dropna(subset=['GP2ID', 'Ancestry'])

# remove duplicated GP2ID within each ancestry
LRRK2_risk_clean = LRRK2_risk_clean.drop_duplicates(subset=['Ancestry', 'GP2ID'])

LRRK2_risk_clean

In [ ]:
LRRK2_risk_clean['Ancestry'].value_counts()

In [ ]:
import os
# base output directory
base_dir = os.path.expanduser("~/workspace/ws_files/r11/cohort")

# save one file for each ancestry
for ancestry, df_anc in LRRK2_risk_clean.groupby('Ancestry'):

    # create ancestry folder
    out_dir = os.path.join(base_dir, ancestry)
    os.makedirs(out_dir, exist_ok=True)

    # make FID and IID columns from GP2ID
    samples_toKeep = pd.DataFrame({
        'FID': df_anc['GP2ID'],
        'IID': df_anc['GP2ID']
    })

    # output file
    out_file = os.path.join(out_dir, "LRRK2risk.samplestoKeep.rm.txt")

    # save without header and without index
    samples_toKeep.to_csv(
        out_file,
        sep='\t',
        index=False,
        header=False
    )

    print(f"{ancestry}: {samples_toKeep.shape[0]} samples saved to {out_file}")

SNCA cohort definition

Annotation of the gene

Extract the region using PLINK

Extract SNCA gene in NBA cohort(excluding tuepac)

SNCA coordinates: Chromosome 4: 89,700,345-89,838,315(GRCh38/hg38)

In [ ]:
## extract region using plink
ancestries = {'EUR','AAC','AFR','AJ','AMR','SAS','EAS','CAS','MDE','CAH'}

for ancestry in ancestries:
    
    WORK_DIR = f'~/workspace/ws_files/r11/cohort/{ancestry}'

    ! /home/jupyter/tools/plink2 \
    --pfile {REL11_PATH}/imputed_genotypes/{ancestry}/chr4_{ancestry}_release11_vwb \
    --chr 4 \
    --from-bp 89700345 \
    --to-bp 89838315 \
    --make-bed \
    --out {WORK_DIR}/{ancestry}_SNCA

PRKN cohort definition

Annotation of the gene

Extract the region using PLINK

Extract PRKN gene in NBA cohort(excluding tuepac)

PRKN coordinates: Chromosome 6: 161,347,417-162,727,775(GRCh38/hg38)

In [ ]:
## extract region using plink
ancestries = {'EUR','AAC','AFR','AJ','AMR','SAS','EAS','CAS','MDE','CAH'}

for ancestry in ancestries:
    
    WORK_DIR = f'~/workspace/ws_files/r11/cohort/{ancestry}'

    ! /home/jupyter/tools/plink2 \
    --pfile {REL11_PATH}/imputed_genotypes/{ancestry}/chr6_{ancestry}_release11_vwb \
    --chr 6 \
    --from-bp 161347417 \
    --to-bp 162727775 \
    --make-bed \
    --out {WORK_DIR}/{ancestry}_PRKN

In [ ]:
## extract variants of interest using plink
ancestries = {'EUR','AAC','AFR','AJ','AMR','SAS','EAS','CAS','MDE','CAH'}

for ancestry in ancestries:
    
    WORK_DIR = f'~/workspace/ws_files/r11/cohort/{ancestry}'

    ! /home/jupyter/tools/plink2 \
    --bfile {WORK_DIR}/{ancestry}_PRKN \
    --extract ~/workspace/ws_files/cohort/variants.list/PRKN.txt \
    --make-bed \
    --out {WORK_DIR}/PRKN

In [ ]:
#--recode A creates a new text fileset, showing each variant in each case and control for the minor allele (A).
# Also extract the significant variants 
ancestries = {'EUR','AAC','AFR','AJ','AMR','SAS','EAS','CAS','MDE','CAH'}

for ancestry in ancestries:
    
    WORK_DIR = f'~/workspace/ws_files/r11/cohort/{ancestry}'

    ! /home/jupyter/tools/plink \
    --bfile {WORK_DIR}/PRKN \
    --keep {WORK_DIR}/{ancestry}.samplestoKeep \
    --extract ~/workspace/ws_files/cohort/variants.list/PRKN.txt \
    --recode A \
    --out {WORK_DIR}/PRKN

In [ ]:
WORK_DIR = f'~/workspace/ws_files/r11/cohort/'

In [ ]:
recode = pd.read_csv(f'{WORK_DIR}/CAH/PRKN.raw', sep='\s+')
recode

In [ ]:
recode_PRKN = recode.copy()
# Define the list of prkn variant column names
PRKN_variants = [
    'chr6:161350125:T:G_G',
    'chr6:161350208:C:T_T',
    'chr6:161785820:G:A_A',
    'chr6:161973317:G:A_A',
    'chr6:162443325:AT:A_A',
    'chr6:162443378:CCT:C_C'
]

# Add the PRKN_status column
recode_PRKN['PRKN_status'] = recode_PRKN[PRKN_variants].apply(
    lambda row: 'PRKNcarriers' if any(val in [1.0, 2.0] for val in row) else '',
    axis=1
)

In [ ]:
# Filter for PRKN risk carriers
PRKN_carriers = recode_PRKN[recode_PRKN['PRKN_status'] == 'PRKNcarriers']
# Count PHENOTYPE values
PRKN_carriers['PHENOTYPE'].value_counts()

In [ ]:
# save sample ID of risk carriers
samples_toKeep = PRKN_carriers[['FID', 'IID']].copy()
samples_toKeep.to_csv(f'~/workspace/ws_files/r11/cohort/CAH/PRKN.samplestoKeep.txt', sep = '\t', index=False, header=None)

RAB32 cohort definition

Annotation of the gene

Extract the region using PLINK

Extract RAB32 gene in NBA cohort(excluding tuepac)

RAB32 coordinates: Chromosome 6: 146,543,833-146,554,953(GRCh38/hg38)

In [ ]:
## extract region using plink
ancestries = {'EUR','AAC','AFR','AJ','AMR','SAS','EAS','CAS','MDE','CAH'}

for ancestry in ancestries:
    
    WORK_DIR = f'~/workspace/ws_files/r11/cohort/{ancestry}'

    ! /home/jupyter/tools/plink2 \
    --pfile {REL11_PATH}/imputed_genotypes/{ancestry}/chr6_{ancestry}_release11_vwb \
    --chr 6 \
    --from-bp 146543833 \
    --to-bp 146554953 \
    --make-bed \
    --out {WORK_DIR}/{ancestry}_RAB32

In [ ]:
## extract variants of interest using plink
ancestries = {'EUR','AAC','AFR','AJ','AMR','SAS','EAS','CAS','MDE','CAH'}

for ancestry in ancestries:
    
    WORK_DIR = f'~/workspace/ws_files/r11/cohort/{ancestry}'

    ! /home/jupyter/tools/plink2 \
    --bfile {WORK_DIR}/{ancestry}_RAB32 \
    --extract ~/workspace/ws_files/cohort/variants.list/RAB32.txt \
    --make-bed \
    --out {WORK_DIR}/RAB32

PINK1 cohort definition

Annotation of the gene

Extract the region using PLINK

Extract PINK1 gene in NBA cohort(excluding tuepac)

PINK1 coordinates: Chromosome 1: 20,633,458-20,651,511(GRCh38/hg38)

In [ ]:
## extract region using plink
ancestries = {'EUR','AAC','AFR','AJ','AMR','SAS','EAS','CAS','MDE','CAH'}

for ancestry in ancestries:
    
    WORK_DIR = f'~/workspace/ws_files/r11/cohort/{ancestry}'

    ! /home/jupyter/tools/plink2 \
    --pfile {REL11_PATH}/imputed_genotypes/{ancestry}/chr1_{ancestry}_release11_vwb \
    --chr 1 \
    --from-bp 20633458 \
    --to-bp 20651511 \
    --make-bed \
    --out {WORK_DIR}/{ancestry}_PINK1

In [ ]:
## extract variants of interest using plink
ancestries = {'EUR','AAC','AFR','AJ','AMR','SAS','EAS','CAS','MDE','CAH'}

for ancestry in ancestries:
    
    WORK_DIR = f'~/workspace/ws_files/r11/cohort/{ancestry}'

    ! /home/jupyter/tools/plink2 \
    --bfile {WORK_DIR}/{ancestry}_PINK1 \
    --extract ~/workspace/ws_files/cohort/variants.list/PINK1.txt \
    --make-bed \
    --out {WORK_DIR}/PINK1

In [ ]:
#--recode A creates a new text fileset, showing each variant in each case and control for the minor allele (A).
# Also extract the significant variants 
ancestries = {'CAS','AMR','EUR','EAS','MDE','AFR'}

for ancestry in ancestries:
    
    WORK_DIR = f'~/workspace/ws_files/r11/cohort/{ancestry}'

    ! /home/jupyter/tools/plink \
    --bfile {WORK_DIR}/PINK1 \
    --keep {WORK_DIR}/{ancestry}.samplestoKeep \
    --extract ~/workspace/ws_files/cohort/variants.list/PINK1.txt \
    --recode A \
    --out {WORK_DIR}/PINK1

In [ ]:
WORK_DIR = f'~/workspace/ws_files/r11/cohort/'

In [ ]:
recode = pd.read_csv(f'{WORK_DIR}/AFR/PINK1.raw', sep='\s+')
recode

In [ ]:
recode_PINK1 = recode.copy()
# Define the list of PINK1 variant column names
PINK1_variants = [
    'chr1:20649217:C:T_T'
]

# Add the PINK1_status column
recode_PINK1['PINK1_status'] = recode_PINK1[PINK1_variants].apply(
    lambda row: 'PINK1carriers' if any(val in [1.0, 2.0] for val in row) else '',
    axis=1
)

In [ ]:
# Filter for pink1 carriers
PINK1_carriers = recode_PINK1[recode_PINK1['PINK1_status'] == 'PINK1carriers']
# Count PHENOTYPE values
PINK1_carriers['PHENOTYPE'].value_counts()

In [ ]:
# save sample ID of risk carriers
samples_toKeep = PINK1_carriers[['FID', 'IID']].copy()
samples_toKeep.to_csv(f'~/workspace/ws_files/r11/cohort/AFR/PINK1.samplestoKeep.txt', sep = '\t', index=False, header=None)

VPS35 cohort definition

Annotation of the gene

Extract the region using PLINK

Extract VPS35 gene in NBA cohort(excluding tuepac)

VPS35 coordinates: Chromosome 16: 46,656,132-46,689,518 (GRCh38/hg38)

In [ ]:
## extract region using plink
ancestries = {'EUR','AAC','AFR','AJ','AMR','SAS','EAS','CAS','MDE','CAH'}

for ancestry in ancestries:
    
    WORK_DIR = f"~/workspace/ws_files/r11/cohort/{ancestry}"

    pfile_path = f"{REL11_PATH}/imputed_genotypes/{ancestry}/chr16_{ancestry}_release11_vwb"
    output_path = f"{WORK_DIR}/{ancestry}_VPS35"

    plink_cmd = f"""
    /home/jupyter/tools/plink2 \
    --pfile {pfile_path} \
    --chr 16 \
    --from-bp 46656132 \
    --to-bp 46689518 \
    --make-bed \
    --out {output_path}
    """

    print(f"Running for ancestry: {ancestry}")
    !{plink_cmd}

In [ ]:
## extract variants of interest using plink
ancestries = {'EUR','AAC','AFR','AJ','AMR','SAS','EAS','CAS','MDE','CAH'}

for ancestry in ancestries:
    
    WORK_DIR = f'~/workspace/ws_files/r11/cohort/{ancestry}'

    ! /home/jupyter/tools/plink2 \
    --bfile {WORK_DIR}/{ancestry}_VPS35 \
    --extract ~/workspace/ws_files/cohort/variants.list/VPS35.txt \
    --make-bed \
    --out {WORK_DIR}/VPS35

DJ1 cohort definition

Annotation of the gene

Extract the region using PLINK

Extract DJ1 gene in NBA cohort(excluding tuepac)

DJ1 coordinates: Chromosome 1: 7,954,291-7,985,505 (GRCh38/hg38)

In [ ]:
WORK_DIR = '~/workspace/ws_files/r11/cohort/'

In [ ]:
## extract region using plink
ancestries = {'EUR','AAC','AFR','AJ','AMR','SAS','EAS','CAS','MDE','CAH'}

for ancestry in ancestries:
    
    WORK_DIR = f'~/workspace/ws_files/r11/cohort/{ancestry}'

    ! /home/jupyter/tools/plink2 \
    --pfile {REL11_PATH}/imputed_genotypes/{ancestry}/chr1_{ancestry}_release11_vwb \
    --chr 1 \
    --from-bp 7954291 \
    --to-bp 7985505 \
    --make-bed \
    --out {WORK_DIR}/{ancestry}_DJ1

In [ ]:
## extract variants of interest using plink
ancestries = {'EUR','AAC','AFR','AJ','AMR','SAS','EAS','CAS','MDE','CAH'}

for ancestry in ancestries:
    
    WORK_DIR = f'~/workspace/ws_files/r11/cohort/{ancestry}'

    ! /home/jupyter/tools/plink2 \
    --bfile {WORK_DIR}/{ancestry}_DJ1 \
    --extract ~/workspace/ws_files/cohort/variants.list/DJ1.txt \
    --make-bed \
    --out {WORK_DIR}/DJ1

In [ ]:
#--recode A creates a new text fileset, showing each variant in each case and control for the minor allele (A).
# Also extract the significant variants 
ancestries = {'EAS','SAS','AFR'}

for ancestry in ancestries:
    
    WORK_DIR = f'~/workspace/ws_files/r11/cohort/{ancestry}'

    ! /home/jupyter/tools/plink \
    --bfile {WORK_DIR}/DJ1 \
    --keep {WORK_DIR}/{ancestry}.samplestoKeep \
    --extract ~/workspace/ws_files/cohort/variants.list/DJ1.txt \
    --recode A \
    --out {WORK_DIR}/DJ1

In [ ]:
WORK_DIR = f'~/workspace/ws_files/r11/cohort/'

In [ ]:
# DJ1
recode = pd.read_csv(f'{WORK_DIR}/AFR/DJ1.raw', sep='\s+')
recode

In [ ]:
recode_DJ1 = recode.copy()
# Define the list of PINK1 variant column names
DJ1_variants = [
    'chr1:7962868:G:A_A'
]

# Add the DJ1_status column
recode_DJ1['DJ1_status'] = recode_DJ1[DJ1_variants].apply(
    lambda row: 'DJ1carriers' if any(val in [1.0, 2.0] for val in row) else '',
    axis=1
)

In [ ]:
# Filter for pink1 carriers
DJ1_carriers = recode_DJ1[recode_DJ1['DJ1_status'] == 'DJ1carriers']
# Count PHENOTYPE values
DJ1_carriers['PHENOTYPE'].value_counts()

In [ ]:
# save sample ID of risk carriers
samples_toKeep = DJ1_carriers[['FID', 'IID']].copy()
samples_toKeep.to_csv(f'~/workspace/ws_files/r11/cohort/AFR/DJ1.samplestoKeep.txt', sep = '\t', index=False, header=None)

ATP13A2 cohort definition

Annotation of the gene

Extract the region using PLINK

Extract ATP13A2 gene in NBA cohort(excluding tuepac)

ATP13A2 coordinates: Chromosome 1: 16,985,958-17,011,928 (GRCh38/hg38)

In [ ]:
## extract region using plink
ancestries = {'EUR','AAC','AFR','AJ','AMR','SAS','EAS','CAS','MDE','CAH'}

for ancestry in ancestries:
    
    WORK_DIR = f'~/workspace/ws_files/r11/cohort/{ancestry}'

    ! /home/jupyter/tools/plink2 \
    --pfile {REL11_PATH}/imputed_genotypes/{ancestry}/chr1_{ancestry}_release11_vwb \
    --chr 1 \
    --from-bp 16985958 \
    --to-bp 17011928 \
    --make-bed \
    --out {WORK_DIR}/{ancestry}_ATP

In [ ]:
## extract variants of interest using plink
ancestries = {'EUR','AAC','AFR','AJ','AMR','SAS','EAS','CAS','MDE','CAH'}

for ancestry in ancestries:
    
    WORK_DIR = f'~/workspace/ws_files/r11/cohort/{ancestry}'

    ! /home/jupyter/tools/plink2 \
    --bfile {WORK_DIR}/{ancestry}_ATP \
    --extract ~/workspace/ws_files/cohort/variants.list/ATP13A2.txt \
    --make-bed \
    --out {WORK_DIR}/ATP

In [ ]:
#--recode A creates a new text fileset, showing each variant in each case and control for the minor allele (A).
# Also extract the significant variants 
ancestries = {'CAS','CAH','EUR','AFR','AJ','AMR'}

for ancestry in ancestries:
    
    WORK_DIR = f'~/workspace/ws_files/r11/cohort/{ancestry}'

    ! /home/jupyter/tools/plink \
    --bfile {WORK_DIR}/ATP \
    --keep {WORK_DIR}/{ancestry}.samplestoKeep \
    --extract ~/workspace/ws_files/cohort/variants.list/ATP13A2.txt \
    --recode A \
    --out {WORK_DIR}/ATP

In [ ]:
WORK_DIR = f'~/workspace/ws_files/r11/cohort/'

In [ ]:
# ATP13A2
recode = pd.read_csv(f'{WORK_DIR}/AFR/ATP.raw', sep='\s+')
recode

In [ ]:
recode_ATP = recode.copy()
# Define the list of ATP13A2 variant column names
ATP_variants = [
    'chr1:16988455:C:T_T',
    'chr1:16989886:C:T_T'
]
# Add the ATP_status column
recode_ATP['ATP_status'] = recode_ATP[ATP_variants].apply(
    lambda row: 'ATPcarriers' if any(val in [1.0, 2.0] for val in row) else '',
    axis=1
)

In [ ]:
# Filter for ATP risk carriers
ATP_carriers = recode_ATP[recode_ATP['ATP_status'] == 'ATPcarriers']
# Count PHENOTYPE values
ATP_carriers['PHENOTYPE'].value_counts()

In [ ]:
# save sample ID of risk carriers
samples_toKeep = ATP_carriers[['FID', 'IID']].copy()
samples_toKeep.to_csv(f'~/workspace/ws_files/r11/cohort/AFR/ATP.samplestoKeep.txt', sep = '\t', index=False, header=None)

DCTN1 cohort definition

Annotation of the gene

Extract the region using PLINK

Extract DCTN1 gene in NBA cohort(excluding tuepac)

DCTN1 coordinates: Chromosome 2: 74,361,154-74,392,087 (GRCh38/hg38)

In [ ]:
## extract region using plink
ancestries = {'EUR','AAC','AFR','AJ','AMR','SAS','EAS','CAS','MDE','CAH'}

for ancestry in ancestries:
    
    WORK_DIR = f'~/workspace/ws_files/r11/cohort/{ancestry}'

    ! /home/jupyter/tools/plink2 \
    --pfile {REL11_PATH}/imputed_genotypes/{ancestry}/chr2_{ancestry}_release11_vwb \
    --chr 2 \
    --from-bp 74361154 \
    --to-bp 74392087 \
    --make-bed \
    --out {WORK_DIR}/{ancestry}_DCTN1

In [ ]:
## extract variants of interest using plink
ancestries = {'EUR','AAC','AFR','AJ','AMR','SAS','EAS','CAS','MDE','CAH'}

for ancestry in ancestries:
    
    WORK_DIR = f'~/workspace/ws_files/r11/cohort/{ancestry}'

    ! /home/jupyter/tools/plink2 \
    --bfile {WORK_DIR}/{ancestry}_DCTN1 \
    --extract ~/workspace/ws_files/cohort/variants.list/DCTN1.txt \
    --make-bed \
    --out {WORK_DIR}/DCTN1

In [ ]:
#--recode A creates a new text fileset, showing each variant in each case and control for the minor allele (A).
# Also extract the significant variants 
ancestries = {'EUR'}

for ancestry in ancestries:
    
    WORK_DIR = f'~/workspace/ws_files/r11/cohort/{ancestry}'

    ! /home/jupyter/tools/plink \
    --bfile {WORK_DIR}/DCTN1 \
    --keep {WORK_DIR}/{ancestry}.samplestoKeep \
    --extract ~/workspace/ws_files/cohort/variants.list/DCTN1.txt \
    --recode A \
    --out {WORK_DIR}/DCTN1

In [ ]:
WORK_DIR = f'~/workspace/ws_files/r11/cohort/'

In [ ]:
# DCTN1
recode = pd.read_csv(f'{WORK_DIR}/EUR/DCTN1.raw', sep='\s+')
recode

In [ ]:
recode_DCTN1 = recode.copy()
# Define the list of DCTN1 variant column names
DCTN1_variants = [
    'chr2:74363626:C:A_A'
]
# Add the DCTN1_status column
recode_DCTN1['DCTN1_status'] = recode_DCTN1[DCTN1_variants].apply(
    lambda row: 'DCTN1carriers' if any(val in [1.0, 2.0] for val in row) else '',
    axis=1
)

In [ ]:
# Filter for DCTN1 risk carriers
DCTN1_carriers = recode_DCTN1[recode_DCTN1['DCTN1_status'] == 'DCTN1carriers']
# Count PHENOTYPE values
DCTN1_carriers['PHENOTYPE'].value_counts()

In [ ]:
# save sample ID of risk carriers
samples_toKeep = DCTN1_carriers[['FID', 'IID']].copy()
samples_toKeep.to_csv(f'~/workspace/ws_files/r11/cohort/EUR/DCTN1.samplestoKeep.txt', sep = '\t', index=False, header=None)

FBXO7 cohort definition

Annotation of the gene

Extract the region using PLINK

Extract FBX07 gene in NBA cohort

FBXO7 coordinates: Chromosome 22: 32,474,676-32,498,829 (GRCh38/hg38)

In [ ]:
## extract region using plink
ancestries = {'EUR','AAC','AFR','AJ','AMR','SAS','EAS','CAS','MDE','CAH'}

for ancestry in ancestries:
    
    WORK_DIR = f'~/workspace/ws_files/r11/cohort/{ancestry}'

    ! /home/jupyter/tools/plink2 \
    --pfile {REL11_PATH}/imputed_genotypes/{ancestry}/chr22_{ancestry}_release11_vwb \
    --chr 22 \
    --from-bp 32474676 \
    --to-bp 32498829 \
    --make-bed \
    --out {WORK_DIR}/{ancestry}_FBXO7

In [ ]:
## extract variants of interest using plink
ancestries = {'EUR','AAC','AFR','AJ','AMR','SAS','EAS','CAS','MDE','CAH'}

for ancestry in ancestries:
    
    WORK_DIR = f'~/workspace/ws_files/r11/cohort/{ancestry}'

    ! /home/jupyter/tools/plink2 \
    --bfile {WORK_DIR}/{ancestry}_FBXO7 \
    --extract ~/workspace/ws_files/cohort/variants.list/FBXO7.txt \
    --make-bed \
    --out {WORK_DIR}/FBXO7

In [ ]:
#--recode A creates a new text fileset, showing each variant in each case and control for the minor allele (A).
# Also extract the significant variants 
ancestries = {'AAC','AFR','EUR','EAS'}

for ancestry in ancestries:
    
    WORK_DIR = f'~/workspace/ws_files/r11/cohort/{ancestry}'

    ! /home/jupyter/tools/plink \
    --bfile {WORK_DIR}/FBXO7 \
    --keep {WORK_DIR}/{ancestry}.samplestoKeep \
    --extract ~/workspace/ws_files/cohort/variants.list/FBXO7.txt \
    --recode A \
    --out {WORK_DIR}/FBXO7

In [ ]:
WORK_DIR = f'~/workspace/ws_files/r11/cohort/'

In [ ]:
# FBXO7
recode = pd.read_csv(f'{WORK_DIR}/AFR/FBXO7.raw', sep='\s+')
recode

In [ ]:
recode_FBX = recode.copy()
# Define the list of FBXO7 variant column names
FBX_variants = [
    'chr22:32498165:C:CAA_CAA'
]
# Add the FBX_status column
recode_FBX['FBX_status'] = recode_FBX[FBX_variants].apply(
    lambda row: 'FBXcarriers' if any(val in [1.0, 2.0] for val in row) else '',
    axis=1
)

In [ ]:
# Filter for FBX risk carriers
FBX_carriers = recode_FBX[recode_FBX['FBX_status'] == 'FBXcarriers']
# Count PHENOTYPE values
FBX_carriers['PHENOTYPE'].value_counts()

In [ ]:
# save sample ID of risk carriers
samples_toKeep = ATP_carriers[['FID', 'IID']].copy()
samples_toKeep.to_csv(f'~/workspace/ws_files/r11/cohort/AFR/FBX.samplestoKeep.txt', sep = '\t', index=False, header=None)

JAM2 cohort definition

Annotation of the gene

Extract the region using PLINK

Extract JAM2 gene in NBA cohort

JAM2 coordinates: Chromosome 21: 25,639,258-25,717,562  (GRCh38/hg38)

In [ ]:
## extract region using plink
ancestries = {'EUR','AAC','AFR','AJ','AMR','SAS','EAS','CAS','MDE','CAH'}

for ancestry in ancestries:
    
    WORK_DIR = f'~/workspace/ws_files/r11/cohort/{ancestry}'

    ! /home/jupyter/tools/plink2 \
    --pfile {REL11_PATH}/imputed_genotypes/{ancestry}/chr21_{ancestry}_release11_vwb \
    --chr 21 \
    --from-bp 25639258 \
    --to-bp 25717562 \
    --make-bed \
    --out {WORK_DIR}/{ancestry}_JAM2

In [ ]:
## extract variants of interest using plink
ancestries = {'EUR','AAC','AFR','AJ','AMR','SAS','EAS','CAS','MDE','CAH'}

for ancestry in ancestries:
    
    WORK_DIR = f'~/workspace/ws_files/r11/cohort/{ancestry}'

    ! /home/jupyter/tools/plink2 \
    --bfile {WORK_DIR}/{ancestry}_JAM2 \
    --extract ~/workspace/ws_files/cohort/variants.list/JAM2.txt \
    --make-bed \
    --out {WORK_DIR}/JAM2

RAB39B cohort definition

Annotation of the gene

Extract the region using PLINK

Extract RAB39B gene in NBA cohort

RAB39B coordinates: Chromosome X: 155,258,235-155,264,491 (GRCh38/hg38)

In [ ]:
## extract region using plink
ancestries = {'EUR','AAC','AFR','AJ','AMR','SAS','EAS','CAS','MDE','CAH'}

for ancestry in ancestries:
    
    WORK_DIR = f'~/workspace/ws_files/r11/cohort/{ancestry}'

    ! /home/jupyter/tools/plink2 \
    --pfile {REL11_PATH}/imputed_genotypes/{ancestry}/chrX_{ancestry}_release11_vwb \
    --chr X \
    --from-bp 155258235 \
    --to-bp 155264491 \
    --make-bed \
    --out {WORK_DIR}/{ancestry}_RAB39B

In [ ]:
## extract variants of interest using plink
ancestries = {'EUR','AAC','AFR','AJ','AMR','SAS','EAS','CAS','MDE','CAH'}

for ancestry in ancestries:
    
    WORK_DIR = f'~/workspace/ws_files/r11/cohort/{ancestry}'

    ! /home/jupyter/tools/plink2 \
    --bfile {WORK_DIR}/{ancestry}_RAB39B \
    --extract ~/workspace/ws_files/cohort/variants.list/RAB39B.txt \
    --make-bed \
    --out {WORK_DIR}/RAB39B

SLC20A2 cohort definition

Annotation of the gene

Extract the region using PLINK

Extract SLC20A2 gene in NBA cohort

SLC20A2 coordinates: Chromosome 8: 42,416,475-42,541,926 (GRCh38/hg38)

In [ ]:
## extract region using plink
ancestries = {'EUR','AAC','AFR','AJ','AMR','SAS','EAS','CAS','MDE','CAH'}

for ancestry in ancestries:
    
    WORK_DIR = f'~/workspace/ws_files/r11/cohort/{ancestry}'

    ! /home/jupyter/tools/plink2 \
    --pfile {REL11_PATH}/imputed_genotypes/{ancestry}/chr8_{ancestry}_release11_vwb \
    --chr 8 \
    --from-bp 42416475 \
    --to-bp 42541926 \
    --make-bed \
    --out {WORK_DIR}/{ancestry}_SLC20A2

In [ ]:
## extract variants of interest using plink
ancestries = {'EUR','AAC','AFR','AJ','AMR','SAS','EAS','CAS','MDE','CAH'}

for ancestry in ancestries:
    
    WORK_DIR = f'~/workspace/ws_files/r11/cohort/{ancestry}'

    ! /home/jupyter/tools/plink2 \
    --bfile {WORK_DIR}/{ancestry}_SLC20A2 \
    --extract ~/workspace/ws_files/cohort/variants.list/SLC20A2.txt \
    --make-bed \
    --out {WORK_DIR}/SLC20A2

SYNJ1 cohort definition

Annotation of the gene

Extract the region using PLINK

Extract SYNJ1 gene in NBA cohort

SYNJ1 coordinates: Chromosome 21: 32,628,759-32,728,040 (GRCh38/hg38)

In [ ]:
## extract region using plink
ancestries = {'EUR','AAC','AFR','AJ','AMR','SAS','EAS','CAS','MDE','CAH'}

for ancestry in ancestries:
    
    WORK_DIR = f'~/workspace/ws_files/r11/cohort/{ancestry}'

    ! /home/jupyter/tools/plink2 \
    --pfile {REL11_PATH}/imputed_genotypes/{ancestry}/chr21_{ancestry}_release11_vwb \
    --chr 21 \
    --from-bp 32628759 \
    --to-bp 32728040 \
    --make-bed \
    --out {WORK_DIR}/{ancestry}_SYNJ1

In [ ]:
## extract variants of interest using plink
ancestries = {'EUR','AAC','AFR','AJ','AMR','SAS','EAS','CAS','MDE','CAH'}

for ancestry in ancestries:
    
    WORK_DIR = f'~/workspace/ws_files/r11/cohort/{ancestry}'

    ! /home/jupyter/tools/plink2 \
    --bfile {WORK_DIR}/{ancestry}_SYNJ1 \
    --extract ~/workspace/ws_files/cohort/variants.list/SYNJ1.txt \
    --make-bed \
    --out {WORK_DIR}/SYNJ1

VPS13C
VPS13C cohort definition

Annotation of the gene

Extract the region using PLINK

Extract VPS13C gene in NBA cohort

VPS13C coordinates: Chromosome 15: 61,852,389-62,060,473 (GRCh38/hg38)

In [ ]:
## extract region using plink
ancestries = {'EUR','AAC','AFR','AJ','AMR','SAS','EAS','CAS','MDE','CAH'}

for ancestry in ancestries:
    
    WORK_DIR = f'~/workspace/ws_files/r11/cohort/{ancestry}'

    ! /home/jupyter/tools/plink2 \
    --pfile {REL11_PATH}/imputed_genotypes/{ancestry}/chr15_{ancestry}_release11_vwb \
    --chr 15 \
    --from-bp 61852389 \
    --to-bp 62060473 \
    --make-bed \
    --out {WORK_DIR}/{ancestry}_VPS13C

In [ ]:
## extract variants of interest using plink
ancestries = {'EUR','AAC','AFR','AJ','AMR','SAS','EAS','CAS','MDE','CAH'}

for ancestry in ancestries:
    
    WORK_DIR = f'~/workspace/ws_files/r11/cohort/{ancestry}'

    ! /home/jupyter/tools/plink2 \
    --bfile {WORK_DIR}/{ancestry}_VPS13C \
    --extract ~/workspace/ws_files/cohort/variants.list/VPS13C.txt \
    --make-bed \
    --out {WORK_DIR}/VPS13C

In [ ]:
#--recode A creates a new text fileset, showing each variant in each case and control for the minor allele (A).
# Also extract the significant variants 
ancestries = {'AAC','AMR','SAS','CAH','EUR','EAS','MDE'}

for ancestry in ancestries:
    
    WORK_DIR = f'~/workspace/ws_files/r11/cohort/{ancestry}'

    ! /home/jupyter/tools/plink \
    --bfile {WORK_DIR}/VPS13C \
    --keep {WORK_DIR}/{ancestry}.samplestoKeep \
    --extract ~/workspace/ws_files/cohort/variants.list/VPS13C.txt \
    --recode A \
    --out {WORK_DIR}/VPS13C

In [ ]:
WORK_DIR = f'~/workspace/ws_files/r11/cohort/'


# VPS13C
recode = pd.read_csv(f'{WORK_DIR}/MDE/VPS13C.raw', sep='\s+')
recode

In [ ]:
recode_VPS13C = recode.copy()
# Define the list of VPS13C variant column names
VPS13C_variants = [
    'chr15:61873386:G:A_A'
]
# Add the VPS13C_status column
recode_VPS13C['VPS13C_status'] = recode_VPS13C[VPS13C_variants].apply(
    lambda row: 'VPS13Ccarriers' if any(val in [1.0, 2.0] for val in row) else '',
    axis=1
)

In [ ]:
# Filter for VPS13C risk carriers
VPS13C_carriers = recode_VPS13C[recode_VPS13C['VPS13C_status'] == 'VPS13Ccarriers']
# Count PHENOTYPE values
VPS13C_carriers['PHENOTYPE'].value_counts()

In [ ]:
# save sample ID of risk carriers
samples_toKeep = VPS13C_carriers[['FID', 'IID']].copy()
samples_toKeep.to_csv(f'~/workspace/ws_files/r11/cohort/AAC/VPS13C.samplestoKeep.txt', sep = '\t', index=False, header=None)

WDR45 cohort definition

Annotation of the gene

Extract the region using PLINK

Extract WDR45 gene in NBA cohort

WDR45 coordinates: Chromosome X: 49,074,433-49,101,170 (GRCh38/hg38)

In [ ]:
## extract region using plink
ancestries = {'EUR','AAC','AFR','AJ','AMR','SAS','EAS','CAS','MDE','CAH'}

for ancestry in ancestries:
    
    WORK_DIR = f'~/workspace/ws_files/r11/cohort/{ancestry}'

    ! /home/jupyter/tools/plink2 \
    --pfile {REL11_PATH}/imputed_genotypes/{ancestry}/chrX_{ancestry}_release11_vwb \
    --chr X \
    --from-bp 49074433 \
    --to-bp 49101170 \
    --make-bed \
    --out {WORK_DIR}/{ancestry}_WDR45

In [ ]:
## extract variants of interest using plink
ancestries = {'EUR','AAC','AFR','AJ','AMR','SAS','EAS','CAS','MDE','CAH'}

for ancestry in ancestries:
    
    WORK_DIR = f'~/workspace/ws_files/r11/cohort/{ancestry}'

    ! /home/jupyter/tools/plink2 \
    --bfile {WORK_DIR}/{ancestry}_WDR45 \
    --extract ~/workspace/ws_files/cohort/variants.list/WDR45.txt \
    --make-bed \
    --out {WORK_DIR}/WDR45

DNAJC6 cohort definition

Annotation of the gene

Extract the region using PLINK

Extract DNAJC6 gene in NBA cohort

DNAJC6 coordinates: Chromosome 1: 65,248,219-65,415,871 (GRCh38/hg38)

In [ ]:
## extract region using plink
ancestries = {'EUR','AAC','AFR','AJ','AMR','SAS','EAS','CAS','MDE','CAH'}

for ancestry in ancestries:
    
    WORK_DIR = f'~/workspace/ws_files/r11/cohort/{ancestry}'

    ! /home/jupyter/tools/plink2 \
    --pfile {REL11_PATH}/imputed_genotypes/{ancestry}/chr1_{ancestry}_release11_vwb \
    --chr 1 \
    --from-bp 65248219 \
    --to-bp 65415871 \
    --make-bed \
    --out {WORK_DIR}/{ancestry}_DNAJC6

In [ ]:
## extract variants of interest using plink
ancestries = {'EUR','AAC','AFR','AJ','AMR','SAS','EAS','CAS','MDE','CAH'}

for ancestry in ancestries:
    
    WORK_DIR = f'~/workspace/ws_files/r11/cohort/{ancestry}'

    ! /home/jupyter/tools/plink2 \
    --bfile {WORK_DIR}/{ancestry}_DNAJC6 \
    --extract ~/workspace/ws_files/cohort/variants.list/DNAJC6.txt \
    --make-bed \
    --out {WORK_DIR}/DNAJC6

Cohort sorting

In [ ]:
WORK_DIR = f'~/workspace/ws_files/r11/cohort/'

In [ ]:
# Find IPD in EUR
GBA_EUR = pd.read_csv(f'~/workspace/ws_files/r11/cohort/EUR/GBA1risk.samplestoKeep.rm.txt', sep='\t', header=None, names=["FID", "IID"])
LRRK2_EUR = pd.read_csv(f'~/workspace/ws_files/r11/cohort/EUR/LRRK2risk.samplestoKeep.rm.txt', sep='\t', header=None, names=["FID", "IID"])
PRKN_EUR = pd.read_csv(f'~/workspace/ws_files/r11/cohort/EUR/PRKN.samplestoKeep.txt', sep='\t', header=None, names=["FID", "IID"])
PINK1_EUR = pd.read_csv(f'~/workspace/ws_files/r11/cohort/EUR/PINK1.samplestoKeep.txt', sep='\t', header=None, names=["FID", "IID"])
VPS13C_EUR = pd.read_csv(f'~/workspace/ws_files/r11/cohort/EUR/VPS13C.samplestoKeep.txt', sep='\t', header=None, names=["FID", "IID"])
ATP_EUR = pd.read_csv(f'~/workspace/ws_files/r11/cohort/EUR/ATP.samplestoKeep.txt', sep='\t', header=None, names=["FID", "IID"])
DCTN1_EUR = pd.read_csv(f'~/workspace/ws_files/r11/cohort/EUR/DCTN1.samplestoKeep.txt', sep='\t', header=None, names=["FID", "IID"])
FBX_EUR = pd.read_csv(f'~/workspace/ws_files/r11/cohort/EUR/FBX.samplestoKeep.txt', sep='\t', header=None, names=["FID", "IID"])

# Merge all 8 DataFrames
merge = pd.concat([GBA_EUR, LRRK2_EUR, PRKN_EUR, PINK1_EUR, VPS13C_EUR, ATP_EUR, DCTN1_EUR, FBX_EUR], ignore_index=True)

# Remove duplicates by IID, keeping the first
merge = merge.drop_duplicates(subset="IID", keep="first")

EUR = pd.read_csv(f'~/workspace/ws_files/r11/cohort/EUR/EUR.samplestoKeep', sep='\t', header=None, names=["FID", "IID"])

# Filter EUR DataFrame to exclude IIDs present in merged_ipd
IPD_EUR = EUR[~EUR['IID'].isin(merge['IID'])]
IPD_EUR

In [ ]:
IPD_EUR.to_csv(f'~/workspace/ws_files/r11/cohort/EUR/IPD_EUR_rm.txt', sep = '\t', index=False, header=None)

In [ ]:
# Find IPD in AAC
GBA_AAC = pd.read_csv(f'~/workspace/ws_files/r11/cohort/AAC/GBA1risk.samplestoKeep.rm.txt', sep='\t', header=None, names=["FID", "IID"])
LRRK2_AAC = pd.read_csv(f'~/workspace/ws_files/r11/cohort/AAC/LRRK2risk.samplestoKeep.rm.txt', sep='\t', header=None, names=["FID", "IID"])
FBXO7_AAC = pd.read_csv(f'~/workspace/ws_files/r11/cohort/AAC/FBX.samplestoKeep.txt', sep='\t', header=None, names=["FID", "IID"])
PRKN_AAC = pd.read_csv(f'~/workspace/ws_files/r11/cohort/AAC/PRKN.samplestoKeep.txt', sep='\t', header=None, names=["FID", "IID"])
VPS13C_AAC = pd.read_csv(f'~/workspace/ws_files/r11/cohort/AAC/VPS13C.samplestoKeep.txt', sep='\t', header=None, names=["FID", "IID"])


# Merge all 4 DataFrames
merge = pd.concat([GBA_AAC, LRRK2_AAC, PRKN_AAC, FBXO7_AAC, VPS13C_AAC], ignore_index=True)

# Remove duplicates by IID, keeping the first
merge = merge.drop_duplicates(subset="IID", keep="first")

AAC = pd.read_csv(f'~/workspace/ws_files/r11/cohort/AAC/AAC.samplestoKeep', sep='\t', header=None, names=["FID", "IID"])

# Filter EUR DataFrame to exclude IIDs present in merged_ipd
IPD_AAC = AAC[~AAC['IID'].isin(merge['IID'])]
IPD_AAC

In [ ]:
IPD_AAC.to_csv(f'~/workspace/ws_files/r11/cohort/AAC/IPD_AAC_rm.txt', sep = '\t', index=False, header=None)

In [ ]:
WORK_DIR = f'~/workspace/ws_files/r11/cohort/AFR'
# Find IPD in AFR
GBA_AFR = pd.read_csv(f'~/workspace/ws_files/r11/cohort/AFR/GBA1risk.samplestoKeep.rm.txt', sep='\t', header=None, names=["FID", "IID"])
LRRK2_AFR = pd.read_csv(f'{WORK_DIR}/LRRK2risk.samplestoKeep.rm.txt', sep='\t', header=None, names=["FID", "IID"])
DJ1_AFR = pd.read_csv(f'{WORK_DIR}/DJ1.samplestoKeep.txt', sep='\t', header=None, names=["FID", "IID"])
PINK1_AFR = pd.read_csv(f'{WORK_DIR}/PINK1.samplestoKeep.txt', sep='\t', header=None, names=["FID", "IID"])
PRKN_AFR = pd.read_csv(f'{WORK_DIR}/PRKN.samplestoKeep.txt', sep='\t', header=None, names=["FID", "IID"])
ATP_AFR = pd.read_csv(f'{WORK_DIR}/ATP.samplestoKeep.txt', sep='\t', header=None, names=["FID", "IID"])
FBX_AFR = pd.read_csv(f'{WORK_DIR}/FBX.samplestoKeep.txt', sep='\t', header=None, names=["FID", "IID"])

# Merge all 6 DataFrames
merge = pd.concat([GBA_AFR, LRRK2_AFR, DJ1_AFR, PINK1_AFR, PRKN_AFR, ATP_AFR, FBX_AFR], ignore_index=True)

# Remove duplicates by IID, keeping the first
merge = merge.drop_duplicates(subset="IID", keep="first")

AFR = pd.read_csv(f'{WORK_DIR}/AFR.samplestoKeep', sep='\t', header=None, names=["FID", "IID"])

# Filter EUR DataFrame to exclude IIDs present in merged_ipd
IPD_AFR = AFR[~AFR['IID'].isin(merge['IID'])]
IPD_AFR

In [ ]:
IPD_AFR.to_csv(f'~/workspace/ws_files/r11/cohort/AFR/IPD_AFR_rm.txt', sep = '\t', index=False, header=None)

In [ ]:
WORK_DIR = f'~/workspace/ws_files/r11/cohort/AJ'
# Find IPD in AJ
GBA_AJ = pd.read_csv(f'~/workspace/ws_files/r11/cohort/AJ/GBA1risk.samplestoKeep.rm.txt', sep='\t', header=None, names=["FID", "IID"])
PRKN_AJ = pd.read_csv(f'{WORK_DIR}/PRKN.samplestoKeep.txt', sep='\t', header=None, names=["FID", "IID"])
LRRK2_AJ = pd.read_csv(f'{WORK_DIR}/LRRK2risk.samplestoKeep.rm.txt', sep='\t', header=None, names=["FID", "IID"])
ATP_AJ = pd.read_csv(f'{WORK_DIR}/ATP.samplestoKeep.txt', sep='\t', header=None, names=["FID", "IID"])

# Merge all 4 DataFrames
merge = pd.concat([GBA_AJ, PRKN_AJ, LRRK2_AJ, ATP_AJ], ignore_index=True)

# Remove duplicates by IID, keeping the first
merge = merge.drop_duplicates(subset="IID", keep="first")

AJ = pd.read_csv(f'{WORK_DIR}/AJ.samplestoKeep', sep='\t', header=None, names=["FID", "IID"])

# Filter EUR DataFrame to exclude IIDs present in merged_ipd
IPD_AJ = AJ[~AJ['IID'].isin(merge['IID'])]
IPD_AJ

In [ ]:
IPD_AJ.to_csv(f'~/workspace/ws_files/r11/cohort/AJ/IPD_AJ_rm.txt', sep = '\t', index=False, header=None)

In [ ]:
WORK_DIR = f'~/workspace/ws_files/r11/cohort/SAS'
# Find IPD in AJ
GBA_SAS = pd.read_csv(f'~/workspace/ws_files/r11/cohort/SAS/GBA1risk.samplestoKeep.rm.txt', sep='\t', header=None, names=["FID", "IID"])
LRRK2_SAS = pd.read_csv(f'{WORK_DIR}/LRRK2risk.samplestoKeep.rm.txt', sep='\t', header=None, names=["FID", "IID"])
PRKN_SAS = pd.read_csv(f'{WORK_DIR}/PRKN.samplestoKeep.txt', sep='\t', header=None, names=["FID", "IID"])
DJ1_SAS = pd.read_csv(f'{WORK_DIR}/DJ1.samplestoKeep.txt', sep='\t', header=None, names=["FID", "IID"])

# Merge all 3 DataFrames
merge = pd.concat([GBA_SAS, LRRK2_SAS, PRKN_SAS, DJ1_SAS], ignore_index=True)

# Remove duplicates by IID, keeping the first
merge = merge.drop_duplicates(subset="IID", keep="first")

SAS = pd.read_csv(f'{WORK_DIR}/SAS.samplestoKeep', sep='\t', header=None, names=["FID", "IID"])

# Filter EUR DataFrame to exclude IIDs present in merged_ipd
IPD_SAS = SAS[~SAS['IID'].isin(merge['IID'])]
IPD_SAS

In [ ]:
IPD_SAS.to_csv(f'~/workspace/ws_files/r11/cohort/SAS/IPD_SAS_rm.txt', sep = '\t', index=False, header=None)

In [ ]:
WORK_DIR = f'~/workspace/ws_files/r11/cohort/AMR'
# Find IPD in AMR
GBA_AMR = pd.read_csv(f'~/workspace/ws_files/r11/cohort/AMR/GBA1risk.samplestoKeep.rm.txt', sep='\t', header=None, names=["FID", "IID"])
PRKN_AMR = pd.read_csv(f'{WORK_DIR}/PRKN.samplestoKeep.txt', sep='\t', header=None, names=["FID", "IID"])
LRRK2_AMR = pd.read_csv(f'{WORK_DIR}/LRRK2risk.samplestoKeep.rm.txt', sep='\t', header=None, names=["FID", "IID"])
PINK1_AMR = pd.read_csv(f'{WORK_DIR}/PINK1.samplestoKeep.txt', sep='\t', header=None, names=["FID", "IID"])
VPS13C_AMR = pd.read_csv(f'{WORK_DIR}/VPS13C.samplestoKeep.txt', sep='\t', header=None, names=["FID", "IID"])
ATP_AMR = pd.read_csv(f'{WORK_DIR}/ATP.samplestoKeep.txt', sep='\t', header=None, names=["FID", "IID"])

# Merge all 6 DataFrames
merge = pd.concat([GBA_AMR, PRKN_AMR, LRRK2_AMR, PINK1_AMR, VPS13C_AMR, ATP_AMR], ignore_index=True)

# Remove duplicates by IID, keeping the first
merge = merge.drop_duplicates(subset="IID", keep="first")

AMR = pd.read_csv(f'{WORK_DIR}/AMR.samplestoKeep', sep='\t', header=None, names=["FID", "IID"])

# Filter AMR DataFrame to exclude IIDs present in merged_ipd
IPD_AMR = AMR[~AMR['IID'].isin(merge['IID'])]
IPD_AMR

In [ ]:
IPD_AMR.to_csv(f'~/workspace/ws_files/r11/cohort/AMR/IPD_AMR_rm.txt', sep = '\t', index=False, header=None)

In [ ]:
# Find IPD in EAS
GBA_EAS = pd.read_csv(f'~/workspace/ws_files/r11/cohort/EAS/GBA1risk.samplestoKeep.rm.txt', sep='\t', header=None, names=["FID", "IID"])
LRRK2_EAS = pd.read_csv(f'~/workspace/ws_files/r11/cohort/EAS/LRRK2risk.samplestoKeep.rm.txt', sep='\t', header=None, names=["FID", "IID"])
PINK1_EAS = pd.read_csv(f'~/workspace/ws_files/r11/cohort/EAS/PINK1.samplestoKeep.txt', sep='\t', header=None, names=["FID", "IID"])
PRKN_EAS = pd.read_csv(f'~/workspace/ws_files/r11/cohort/EAS/PRKN.samplestoKeep.txt', sep='\t', header=None, names=["FID", "IID"])
DJ1_EAS = pd.read_csv(f'~/workspace/ws_files/r11/cohort/EAS/DJ1.samplestoKeep.txt', sep='\t', header=None, names=["FID", "IID"])


# Merge all 5 DataFrames
merge = pd.concat([GBA_EAS, LRRK2_EAS, PRKN_EAS, PINK1_EAS, DJ1_EAS], ignore_index=True)

# Remove duplicates by IID, keeping the first
merge = merge.drop_duplicates(subset="IID", keep="first")

EAS = pd.read_csv(f'~/workspace/ws_files/r11/cohort/EAS/EAS.samplestoKeep', sep='\t', header=None, names=["FID", "IID"])

# Filter EAS DataFrame to exclude IIDs present in merged_ipd
IPD_EAS = EAS[~EAS['IID'].isin(merge['IID'])]
IPD_EAS

In [ ]:
IPD_EAS.to_csv(f'~/workspace/ws_files/r11/cohort/EAS/IPD_EAS_rm.txt', sep = '\t', index=False, header=None)

In [ ]:
# Find IPD in CAS
GBA_CAS = pd.read_csv(f'~/workspace/ws_files/r11/cohort/CAS/GBA1risk.samplestoKeep.rm.txt', sep='\t', header=None, names=["FID", "IID"])
LRRK2_CAS = pd.read_csv(f'~/workspace/ws_files/r11/cohort/CAS/LRRK2risk.samplestoKeep.rm.txt', sep='\t', header=None, names=["FID", "IID"])
PRKN_CAS = pd.read_csv(f'~/workspace/ws_files/r11/cohort/CAS/PRKN.samplestoKeep.txt', sep='\t', header=None, names=["FID", "IID"])
PINK1_CAS = pd.read_csv(f'~/workspace/ws_files/r11/cohort/CAS/PINK1.samplestoKeep.txt', sep='\t', header=None, names=["FID", "IID"])

# Merge all 4 DataFrames
merge = pd.concat([GBA_CAS, LRRK2_CAS, PRKN_CAS, PINK1_CAS], ignore_index=True)

# Remove duplicates by IID, keeping the first
merge = merge.drop_duplicates(subset="IID", keep="first")

CAS = pd.read_csv(f'~/workspace/ws_files/r11/cohort/CAS/CAS.samplestoKeep', sep='\t', header=None, names=["FID", "IID"])

# Filter CAS DataFrame to exclude IIDs present in merged_ipd
IPD_CAS = CAS[~CAS['IID'].isin(merge['IID'])]
IPD_CAS

In [ ]:
IPD_CAS.to_csv(f'~/workspace/ws_files/r11/cohort/CAS/IPD_CAS_rm.txt', sep = '\t', index=False, header=None)

In [ ]:
# Find IPD in MDE
GBA_MDE = pd.read_csv(f'~/workspace/ws_files/r11/cohort/MDE/GBA1risk.samplestoKeep.rm.txt', sep='\t', header=None, names=["FID", "IID"])
LRRK2_MDE = pd.read_csv(f'~/workspace/ws_files/r11/cohort/MDE/LRRK2risk.samplestoKeep.rm.txt', sep='\t', header=None, names=["FID", "IID"])
PRKN_MDE = pd.read_csv(f'~/workspace/ws_files/r11/cohort/MDE/PRKN.samplestoKeep.txt', sep='\t', header=None, names=["FID", "IID"])
PINK1_MDE = pd.read_csv(f'~/workspace/ws_files/r11/cohort/MDE/PINK1.samplestoKeep.txt', sep='\t', header=None, names=["FID", "IID"])

# Merge all 4 DataFrames
merge = pd.concat([GBA_MDE, LRRK2_MDE, PRKN_MDE, PINK1_MDE], ignore_index=True)

# Remove duplicates by IID, keeping the first
merge = merge.drop_duplicates(subset="IID", keep="first")

MDE = pd.read_csv(f'~/workspace/ws_files/r11/cohort/MDE/MDE.samplestoKeep', sep='\t', header=None, names=["FID", "IID"])

# Filter MDE DataFrame to exclude IIDs present in merged_ipd
IPD_MDE = MDE[~MDE['IID'].isin(merge['IID'])]
IPD_MDE

In [ ]:
IPD_MDE.to_csv(f'~/workspace/ws_files/r11/cohort/MDE/IPD_MDE_rm.txt', sep = '\t', index=False, header=None)

In [ ]:
# Find IPD in CAH
GBA_CAH = pd.read_csv(f'~/workspace/ws_files/r11/cohort/CAH/GBA1risk.samplestoKeep.rm.txt', sep='\t', header=None, names=["FID", "IID"])
LRRK2_CAH = pd.read_csv(f'~/workspace/ws_files/r11/cohort/CAH/LRRK2risk.samplestoKeep.rm.txt', sep='\t', header=None, names=["FID", "IID"])
PRKN_CAH = pd.read_csv(f'~/workspace/ws_files/r11/cohort/CAH/PRKN.samplestoKeep.txt', sep='\t', header=None, names=["FID", "IID"])
ATP_CAH = pd.read_csv(f'~/workspace/ws_files/r11/cohort/CAH/ATP.samplestoKeep.txt', sep='\t', header=None, names=["FID", "IID"])
VPS13C_CAH = pd.read_csv(f'~/workspace/ws_files/r11/cohort/CAH/VPS13C.samplestoKeep.txt', sep='\t', header=None, names=["FID", "IID"])


# Merge all 4 DataFrames
merge = pd.concat([GBA_CAH, LRRK2_CAH, PRKN_CAH, ATP_CAH, VPS13C_CAH], ignore_index=True)

# Remove duplicates by IID, keeping the first
merge = merge.drop_duplicates(subset="IID", keep="first")

CAH = pd.read_csv(f'~/workspace/ws_files/r11/cohort/CAH/CAH.samplestoKeep', sep='\t', header=None, names=["FID", "IID"])

# Filter CAH DataFrame to exclude IIDs present in merged_ipd
IPD_CAH = CAH[~CAH['IID'].isin(merge['IID'])]
IPD_CAH

In [ ]:
IPD_CAH.to_csv(f'~/workspace/ws_files/r11/cohort/CAH/IPD_CAH_rm.txt', sep = '\t', index=False, header=None)